In [ ]:
import sys
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import euclidean_distances
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
import torch
from torch.utils.data import TensorDataset, DataLoader
from google.colab import drive


In [ ]:
def random_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

random_seed(1234)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add dataset directory to path
import sys
sys.path.append('/content/drive/MyDrive/Collected_dataset')

# OS and path setup
import os
ospj = os.path.join
path_workspace = '/content/drive/MyDrive/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define training data paths
dict_path_tr_clean = {
    #'20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
    # '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
    #'50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
    # '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
    #'70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
    # '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
}


dict_path_te_clean = {
    # '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
    # '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
}



# dict_path_tr_clean = {
#     '20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
#     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
#     '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
#     '50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
#     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
#     '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
#     '70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
#     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
#     '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
# }


# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
#     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
#     '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
# }






# dict_path_tr_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/35m_stat_train.csv'),
# }
# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat_test.csv'),
# }

# 우선
# 20 ,50 ,70 stat 만 file , 1 개씩 valid 35 stat test 로 파일 구성
# training 하고 test 는 다를수있어도 valid 랑 test 는 유사한지 확인
# baseline 모델 이랑 lstm 구현 epoch 은 100 만 수요일 4시반 목요일대신





In [ ]:
import pandas as pd

# Select specific columns
selected_columns = ['lat', 'lon', 'alt', 'alt_ellipsoid','vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'timestamp']

# Initialize dictionaries
dict_df_tr_clean = {}
dict_df_te_clean = {}

# Load training CSVs
for key, csv_path in dict_path_tr_clean.items():
    try:
        df = pd.read_csv(csv_path, usecols=selected_columns)
        dict_df_tr_clean[key] = df
        print(f"{key} ✅ Training data loaded")
    except Exception as e:
        print(f"{key} ❌ Training load error: {e}")

# Load testing CSVs
for key, csv_path in dict_path_te_clean.items():
    try:
        df = pd.read_csv(csv_path, usecols=selected_columns)
        dict_df_te_clean[key] = df
        print(f"{key} ✅ Test data loaded")
    except Exception as e:
        print(f"{key} ❌ Test load error: {e}")


20m_dyn1 ✅ Training data loaded
50m_dyn1 ✅ Training data loaded
70m_dyn1 ✅ Training data loaded
35m_dyn1 ✅ Test data loaded


In [ ]:
def convert_timestamp_to_seconds(t):
    try:
        minutes, sec_fraction = t.split(":")
        return int(minutes) * 60 + float(sec_fraction)
    except Exception as e:
        print(f"Error converting timestamp: {t} -> {e}")
        return None


In [ ]:
def split_by_time_gap(df, time_col='timestamp', threshold=1.0):
    df = df.copy()
    df[time_col] = df[time_col].apply(convert_timestamp_to_seconds)
    df = df.dropna(subset=[time_col]).reset_index(drop=True)

    print("\n🧪 Converted timestamp (head):")
    print(df[time_col].iloc[:5].tolist())

    diffs = df[time_col].diff().fillna(0).abs()
    split_indices = diffs[diffs >= threshold].index.tolist()

    split_indices = [0] + split_indices + [len(df)]
    splits = [
        df.iloc[split_indices[i]:split_indices[i + 1]].reset_index(drop=True)
        for i in range(len(split_indices) - 1)
    ]
    return splits


In [ ]:
# Apply segment split to training and testing data
dict_split_tr_clean = {}
dict_split_te_clean = {}

for key, df in dict_df_tr_clean.items():
    split_dfs = split_by_time_gap(df, time_col='timestamp', threshold=1.0)
    dict_split_tr_clean[key] = split_dfs
    print(f"{key} ✅ Segmented - {len(split_dfs)} segments")

for key, df in dict_df_te_clean.items():
    split_dfs = split_by_time_gap(df, time_col='timestamp', threshold=1.0)
    dict_split_te_clean[key] = split_dfs
    print(f"{key} ✅ Segmented - {len(split_dfs)} segments")



🧪 Converted timestamp (head):
[270.4, 270.6, 270.8, 271.0, 271.2]
20m_dyn1 ✅ Segmented - 4 segments

🧪 Converted timestamp (head):
[256.0, 256.2, 256.4, 256.6, 256.8]
50m_dyn1 ✅ Segmented - 4 segments

🧪 Converted timestamp (head):
[330.6, 330.8, 331.0, 331.2, 331.4]
70m_dyn1 ✅ Segmented - 5 segments

🧪 Converted timestamp (head):
[226.7, 226.9, 227.1, 227.3, 227.5]
35m_dyn1 ✅ Segmented - 4 segments


In [ ]:
dict_split_train_only = {}
dict_split_valid_only = {}

for key, segments in dict_split_tr_clean.items():
    if len(segments) > 1:
        dict_split_train_only[key] = segments[:-1]
        dict_split_valid_only[key] = [segments[-1]]
    else:
        dict_split_train_only[key] = []
        dict_split_valid_only[key] = segments

# 확인
for k in dict_split_tr_clean.keys():
    print(f"{k} → Train seg: {len(dict_split_train_only[k])}, Valid seg: {len(dict_split_valid_only[k])}")

20m_dyn1 → Train seg: 3, Valid seg: 1
50m_dyn1 → Train seg: 3, Valid seg: 1
70m_dyn1 → Train seg: 4, Valid seg: 1


In [ ]:
def count_total_points(dict_original, dict_split):
    for key in dict_original:
        original_count = len(dict_original[key])
        split_count = sum(len(df) for df in dict_split[key])
        status = "✅ 일치" if original_count == split_count else "❌ 불일치"
        print(f"{key}: 원본 = {original_count} / 분할 합계 = {split_count} → {status}")




In [ ]:

# =====================================
# 3. 원본 vs Segment 개수 비교
# =====================================
print("\n🔎 학습 데이터 확인 (원본 vs 전체 segment)")
count_total_points(dict_df_tr_clean, dict_split_tr_clean)

print("\n🔎 테스트 데이터 확인 (원본 vs 전체 segment)")
count_total_points(dict_df_te_clean, dict_split_te_clean)



🔎 학습 데이터 확인 (원본 vs 전체 segment)
20m_dyn1: 원본 = 1826 / 분할 합계 = 1826 → ✅ 일치
50m_dyn1: 원본 = 1980 / 분할 합계 = 1980 → ✅ 일치
70m_dyn1: 원본 = 2000 / 분할 합계 = 2000 → ✅ 일치

🔎 테스트 데이터 확인 (원본 vs 전체 segment)
35m_dyn1: 원본 = 2120 / 분할 합계 = 2120 → ✅ 일치


In [ ]:
# =====================================
# 4. Train / Valid 포인트 수 요약
# =====================================
train_points = sum(len(df) for dfs in dict_split_train_only.values() for df in dfs)
valid_points = sum(len(df) for dfs in dict_split_valid_only.values() for df in dfs)
test_points  = sum(len(df) for dfs in dict_split_te_clean.values()   for df in dfs)

print("\n📊 데이터 포인트 개수 요약")
print(f"Train total points: {train_points}")
print(f"Valid total points: {valid_points}")
print(f"Test total points:  {test_points}")


📊 데이터 포인트 개수 요약
Train total points: 4444
Valid total points: 1362
Test total points:  2120


In [ ]:
def add_delta_coordinates(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            df['D_lat'] = [0.0] + (df['lat'].diff().fillna(0).tolist())[1:]
            df['D_lon'] = [0.0] + (df['lon'].diff().fillna(0).tolist())[1:]
            df['D_alt'] = [0.0] + (df['alt'].diff().fillna(0).tolist())[1:]


In [ ]:
from sklearn.metrics import euclidean_distances

def calculate_euc_distance(df):
    coordinates = df[['lat', 'lon', 'alt']].to_numpy()
    distances = [0.0]
    for i in range(1, len(coordinates)):
        dist = euclidean_distances([coordinates[i - 1]], [coordinates[i]])[0][0]
        distances.append(dist)
    return distances

def add_euc_distance(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            distances = calculate_euc_distance(df)
            df['d_pos'] = distances


In [ ]:
def add_time_gap(dict_split):
    for key, split_list in dict_split.items():
        for df in split_list:
            time_gap = [0.0] + (df['timestamp'].diff().fillna(0).tolist())[1:]
            df['time_gap'] = time_gap


In [ ]:
# Apply feature creation to both training and test sets
add_delta_coordinates(dict_split_train_only)
add_delta_coordinates(dict_split_valid_only)
add_delta_coordinates(dict_split_te_clean)

add_euc_distance(dict_split_train_only)
add_euc_distance(dict_split_valid_only)
add_euc_distance(dict_split_te_clean)

add_time_gap(dict_split_train_only)
add_time_gap(dict_split_valid_only)
add_time_gap(dict_split_te_clean)


In [ ]:
# =====================================
# 세트별 segment 개수 더블 체크
# =====================================
print("\n📦 Segment 개수 확인")
print(f"Train set: {sum(len(segments) for segments in dict_split_train_only.values())} segments total")
print(f"Valid set: {sum(len(segments) for segments in dict_split_valid_only.values())} segments total")
print(f"Test set:  {sum(len(segments) for segments in dict_split_te_clean.values())} segments total")

# key별 segment 개수 확인 (원하면 주석 해제)
for k in dict_split_train_only.keys():
    print(f"{k} → Train seg: {len(dict_split_train_only[k])}, Valid seg: {len(dict_split_valid_only[k])}")



📦 Segment 개수 확인
Train set: 10 segments total
Valid set: 3 segments total
Test set:  4 segments total
20m_dyn1 → Train seg: 3, Valid seg: 1
50m_dyn1 → Train seg: 3, Valid seg: 1
70m_dyn1 → Train seg: 4, Valid seg: 1


In [ ]:
# # Extract rows where time_gap < 0
# neg_time_gap_rows = dict_split_tr_clean['20m_stat'][0][dict_split_tr_clean['20m_stat'][0]['time_gap'] < 0]

# # Display them
# print("❗ Rows with time_gap < 0:")
# print(neg_time_gap_rows)


In [ ]:
import numpy as np

# Define window size
window_size = 5 # 기본 5개
# Real 5개를 가지고 6번째 예측
#2,3,4,5,6 을보고 7을 예측

# Define input features and target features
input_features =  ['vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'D_lat', 'D_lon', 'D_alt', 'd_pos','time_gap']
# 'vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'D_lat', 'D_lon', 'D_alt', 'd_pos','time_gap'

target_features = ['D_alt']  # target to predict


In [ ]:
def create_sequence_label_tensors(dict_split, window_size):
    X_all, y_all = [], []
    for df_list in dict_split.values():
        for df in df_list:
            if len(df) <= window_size:
                continue
            data = df[input_features].to_numpy()
            labels = df[target_features].to_numpy()
            for i in range(len(df) - window_size):
                window = data[i : i + window_size]
                label = labels[i + window_size]
                X_all.append(window)
                y_all.append(label)
    X_tensor = np.array(X_all)
    y_tensor = np.array(y_all)
    return X_tensor, y_tensor

In [ ]:
# Apply sequence generation
X_train, y_train = create_sequence_label_tensors(dict_split_train_only, window_size)
X_valid, y_valid = create_sequence_label_tensors(dict_split_valid_only, window_size)
X_test,  y_test  = create_sequence_label_tensors(dict_split_te_clean,  window_size)


In [ ]:
# Check resulting shapes
print(f"✅ Train sequences shape: {X_train.shape}")
print(f"✅ Train labels shape:    {y_train.shape}")

print(f"✅ Valid sequences shape: {X_valid.shape}")
print(f"✅ Valid labels shape:    {y_valid.shape}")

print(f"✅ Test sequences shape:  {X_test.shape}")
print(f"✅ Test labels shape:     {y_test.shape}")

✅ Train sequences shape: (4394, 5, 10)
✅ Train labels shape:    (4394, 1)
✅ Valid sequences shape: (1347, 5, 10)
✅ Valid labels shape:    (1347, 1)
✅ Test sequences shape:  (2100, 5, 10)
✅ Test labels shape:     (2100, 1)


In [ ]:
def count_expected_windows(dict_split, window_size):
    expected_total = 0
    for df_list in dict_split.values():
        for df in df_list:
            expected_total += max(0, len(df) - window_size)
    return expected_total

# ✅ 세트별 기대 윈도우 개수
expected_train = count_expected_windows(dict_split_train_only, window_size)
expected_valid = count_expected_windows(dict_split_valid_only, window_size)
expected_test  = count_expected_windows(dict_split_te_clean,  window_size)

# ✅ 실제 생성된 윈도우 개수 (앞서 만든 넘파이 배열 기준)
actual_train = X_train.shape[0]
actual_valid = X_valid.shape[0]
actual_test  = X_test.shape[0]

# ✅ 매칭 여부 출력
print(f"✅ Train: expected = {expected_train}, actual = {actual_train} → {'✔️ Match' if expected_train == actual_train else '❌ Mismatch'}")
print(f"✅ Valid: expected = {expected_valid}, actual = {actual_valid} → {'✔️ Match' if expected_valid == actual_valid else '❌ Mismatch'}")
print(f"✅ Test:  expected = {expected_test},  actual = {actual_test}  → {'✔️ Match' if expected_test  == actual_test  else '❌ Mismatch'}")


✅ Train: expected = 4394, actual = 4394 → ✔️ Match
✅ Valid: expected = 1347, actual = 1347 → ✔️ Match
✅ Test:  expected = 2100,  actual = 2100  → ✔️ Match


In [ ]:
def create_sequence_label_tensors(dict_split, window_size):
    X_all, y_all = [], []
    for df_list in dict_split.values():
        for df in df_list:
            if len(df) <= window_size:
                continue
            data = df[input_features].to_numpy()
            labels = df[target_features].to_numpy()
            for i in range(len(df) - window_size):
                window = data[i : i + window_size]
                label = labels[i + window_size]
                X_all.append(window)
                y_all.append(label)
    X_tensor = np.array(X_all)
    y_tensor = np.array(y_all)
    return X_tensor, y_tensor

In [ ]:
X_train, y_train = create_sequence_label_tensors(dict_split_train_only, window_size)
X_valid, y_valid = create_sequence_label_tensors(dict_split_valid_only, window_size)
X_test,  y_test  = create_sequence_label_tensors(dict_split_te_clean,  window_size)


In [ ]:
print(f"✅ Train sequences shape: {X_train.shape}")
print(f"✅ Train labels shape:    {y_train.shape}")
print(f"✅ Valid sequences shape: {X_valid.shape}")
print(f"✅ Valid labels shape:    {y_valid.shape}")
print(f"✅ Test sequences shape:  {X_test.shape}")
print(f"✅ Test labels shape:     {y_test.shape}")

✅ Train sequences shape: (4394, 5, 10)
✅ Train labels shape:    (4394, 1)
✅ Valid sequences shape: (1347, 5, 10)
✅ Valid labels shape:    (1347, 1)
✅ Test sequences shape:  (2100, 5, 10)
✅ Test labels shape:     (2100, 1)


In [ ]:
def make_loader(X, y, batch_size=64, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
# Train loader
train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)

# Valid loader
X_val_tensor = torch.tensor(X_valid, dtype=torch.float32)
y_val_tensor = torch.tensor(y_valid, dtype=torch.float32)
valid_dataset = TensorDataset(X_val_tensor, y_val_tensor)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

# Test loader
X_te_tensor = torch.tensor(X_test, dtype=torch.float32)
y_te_tensor = torch.tensor(y_test, dtype=torch.float32)
test_dataset = TensorDataset(X_te_tensor, y_te_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error

def compute_all_metrics(preds, targets):
    # 텐서를 numpy로 변환 (GPU/grad 대응)
    if isinstance(preds, torch.Tensor):
        preds = preds.detach().cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.detach().cpu().numpy()

    # sklearn 기본 MSE (전체 좌표 포함 평균)
    mse = mean_squared_error(targets, preds)

    return {
        'MSE': mse
    }


In [ ]:
def show_delta_predictions(preds, targets, n=10):
    if isinstance(preds, torch.Tensor):
        preds = preds.detach().cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.detach().cpu().numpy()
    df = pd.DataFrame({
#        'D_lat_pred': preds[:n, 0],
#        'D_lat_true': targets[:n, 0],
        'D_alt_pred': preds[:n, 0],
        'D_alt_true': targets[:n, 0],
#        'D_lon_pred': preds[:n, 1],
#        'D_lon_true': targets[:n, 1],
#        'D_alt_pred': preds[:n, 2],
#        'D_alt_true': targets[:n, 2],
    })
    return df.round(4)


In [ ]:
import torch
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, lstm_hidden_dim, lstm_layers, mlp_layers, output_dim=2, dropout=0.2):
        super(LSTMRegressor, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=lstm_hidden_dim,
                            num_layers=lstm_layers,
                            batch_first=True,
                            dropout=dropout)
        mlp = []
        in_dim = lstm_hidden_dim
        for _ in range(mlp_layers):
            out_dim = max(in_dim // 2, 8)
            mlp.append(nn.Linear(in_dim, out_dim))
            mlp.append(nn.ReLU())
            in_dim = out_dim
        mlp.append(nn.Linear(in_dim, output_dim))
        self.mlp = nn.Sequential(*mlp)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)        # (batch, seq_len, hidden)
        last_hidden = lstm_out[:, -1, :]  # 마지막 time step의 hidden
        return self.mlp(last_hidden)


In [ ]:
import torch
import torch.nn as nn

def train_model_with_valid_early_stop(
    input_dim,
    output_dim,
    train_loader,
    valid_loader,
    n_epochs=300,
    lr=0.001,
    lstm_hidden_dim=128,
    lstm_layers=2,
    mlp_layers=4,
    patience=30,
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    model = LSTMRegressor(
        input_dim=input_dim,
        lstm_hidden_dim=lstm_hidden_dim,
        lstm_layers=lstm_layers,
        mlp_layers=mlp_layers,
        output_dim=output_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    train_mse_curve, valid_mse_curve = [], []
    best_mse = float('inf')
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(n_epochs):
        # === Train ===
        model.train()
        all_preds, all_targets = [], []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            all_preds.append(preds.detach().cpu())
            all_targets.append(y_batch.detach().cpu())

        train_mse = compute_all_metrics(torch.cat(all_preds), torch.cat(all_targets))['MSE']
        train_mse_curve.append(train_mse)

        # === Validation ===
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                val_preds.append(preds.detach().cpu())
                val_targets.append(y_batch.detach().cpu())
        valid_mse = compute_all_metrics(torch.cat(val_preds), torch.cat(val_targets))['MSE']
        valid_mse_curve.append(valid_mse)

        # 로그 출력
        if (epoch + 1) % 5 == 0 or valid_mse < best_mse:
            print(f"Epoch {epoch+1}/{n_epochs} - "
                  f"Train MSE: {train_mse:.6f} | Valid MSE: {valid_mse:.6f} | "
                  f"Best Valid MSE: {best_mse:.6f} (Epoch {best_epoch+1})")

        # Early stopping (valid 기준)
        if valid_mse < best_mse:
            best_mse = valid_mse
            best_state = model.state_dict()
            best_epoch = epoch
            no_improve = 0
            print("🎯 New Best Valid MSE!")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
                break

    # 최적 가중치 로드 (best_state가 None이 아닌 경우에만)
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"✅ Finished. Best Valid MSE: {best_mse:.6f} at Epoch {best_epoch+1}")

    return model, train_mse_curve, valid_mse_curve, best_mse, best_epoch


In [ ]:
import torch
import numpy as np

def evaluate_on_test_set_direct(
    model,
    test_loader,
    device=None,
    show_samples=10
):
    """
    이미 학습된 모델(model)을 그대로 사용해 test_loader를 평가합니다.
    - 모델 저장/재로딩 없이 바로 평가
    - compute_all_metrics를 사용해 지표(MSE 등) 계산
    """
    # 모델이 올라가 있는 디바이스 자동 감지
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu()  # CPU로 모으기
            y_batch = y_batch.cpu()
            all_preds.append(preds)
            all_targets.append(y_batch)

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    # 지표 계산
    test_metrics = compute_all_metrics(all_preds, all_targets)

    # 로그 출력
    print("\n🎯 Test Set Evaluation (direct)")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.6f}")

    # 예측 vs 실제 샘플 표시
    print("\n🔍 Sample Predictions vs Targets:")
    preds_np = all_preds.numpy()
    targets_np = all_targets.numpy()
    for i in range(min(show_samples, len(preds_np))):
        print(f"[{i}] pred: {preds_np[i]},  true: {targets_np[i]}")

    return test_metrics, all_preds, all_targets


In [ ]:
# 하이퍼파라미터
lstm_hidden_dim = 128
lstm_layers = 2
mlp_layers = 3
best_params = [lstm_hidden_dim, lstm_layers, mlp_layers]  # (재현/로그용)


In [ ]:
model, tr_curve, va_curve, best_mse, best_ep = train_model_with_valid_early_stop(
    input_dim=len(input_features),          # 8
    output_dim=len(target_features),        # 2 (D_lat, D_lon)
    train_loader=train_loader,
    valid_loader=valid_loader,
    n_epochs=1000,
    lr=0.0001,
    lstm_hidden_dim=lstm_hidden_dim,
    lstm_layers=lstm_layers,
    mlp_layers=mlp_layers,
    patience=200
)


Epoch 1/1000 - Train MSE: 4064.189453 | Valid MSE: 4830.084473 | Best Valid MSE: inf (Epoch 1)
🎯 New Best Valid MSE!
Epoch 2/1000 - Train MSE: 4051.976318 | Valid MSE: 4796.923340 | Best Valid MSE: 4830.084473 (Epoch 1)
🎯 New Best Valid MSE!
Epoch 3/1000 - Train MSE: 3998.029053 | Valid MSE: 4710.066895 | Best Valid MSE: 4796.923340 (Epoch 2)
🎯 New Best Valid MSE!
Epoch 4/1000 - Train MSE: 3897.823486 | Valid MSE: 4567.757812 | Best Valid MSE: 4710.066895 (Epoch 3)
🎯 New Best Valid MSE!
Epoch 5/1000 - Train MSE: 3740.845703 | Valid MSE: 4354.652832 | Best Valid MSE: 4567.757812 (Epoch 4)
🎯 New Best Valid MSE!
Epoch 6/1000 - Train MSE: 3533.447021 | Valid MSE: 4071.368896 | Best Valid MSE: 4354.652832 (Epoch 5)
🎯 New Best Valid MSE!
Epoch 7/1000 - Train MSE: 3276.968506 | Valid MSE: 3737.249756 | Best Valid MSE: 4071.368896 (Epoch 6)
🎯 New Best Valid MSE!
Epoch 8/1000 - Train MSE: 2985.548340 | Valid MSE: 3358.619873 | Best Valid MSE: 3737.249756 (Epoch 7)
🎯 New Best Valid MSE!
Epoch 9/

In [ ]:
test_metrics, test_preds, test_targets = evaluate_on_test_set_direct(
    model=model,
    test_loader=test_loader,
    device=None,       # 모델이 올라간 디바이스 자동 사용
    show_samples=10
)

# 🔍 내가 만든 함수로 예측 결과 보기
show_delta_predictions(test_preds, test_targets, n=10)



🎯 Test Set Evaluation (direct)
MSE: 851.897522

🔍 Sample Predictions vs Targets:
[0] pred: [-3.4953325],  true: [-1.]
[1] pred: [-2.5674894],  true: [-6.]
[2] pred: [-26.4837],  true: [6.]
[3] pred: [-19.218483],  true: [-9.]
[4] pred: [-27.507858],  true: [6.]
[5] pred: [-15.260251],  true: [-17.]
[6] pred: [-33.516277],  true: [-18.]
[7] pred: [-18.608418],  true: [18.]
[8] pred: [-7.294115],  true: [19.]
[9] pred: [0.57728785],  true: [9.]


,D_alt_pred,D_alt_true
0,-3.4953,-1.0
1,-2.5675,-6.0
2,-26.4837,6.0
3,-19.2185,-9.0
4,-27.5079,6.0
5,-15.2603,-17.0
6,-33.5163,-18.0
7,-18.6084,18.0
8,-7.2941,19.0
9,0.5773,9.0


In [ ]:
# # ==== Valid / Test 예측값을 뽑아 CSV로 저장 ====
# import pandas as pd
# import torch
# import os

# def collect_preds(model, loader, device=None):
#     """loader 전체에 대해 예측값과 실제값을 numpy로 반환"""
#     if device is None:
#         device = next(model.parameters()).device
#     model.eval()
#     preds, trues = [], []
#     with torch.no_grad():
#         for Xb, yb in loader:
#             Xb = Xb.to(device)
#             out = model(Xb).cpu()
#             preds.append(out)
#             trues.append(yb.cpu())
#     preds = torch.cat(preds, dim=0).numpy()
#     trues = torch.cat(trues, dim=0).numpy()
#     return preds, trues

# # 1) Validation 결과 저장
# val_preds, val_trues = collect_preds(model, valid_loader)
# df_valid = pd.DataFrame({
#     "D_lat_pred": val_preds[:, 0],
#     "D_lat_true": val_trues[:, 0],
#     "D_lon_pred": val_preds[:, 1],
#     "D_lon_true": val_trues[:, 1],
# })
# # Colab 로컬(세션) 저장
# df_valid.to_csv("/content/valid_results.csv", index=False)
# # 구글 드라이브에도 저장 (마운트되어 있음)
# df_valid.to_csv("/content/drive/MyDrive/valid_results.csv", index=False)
# print("✅ Saved: /content/valid_results.csv & /content/drive/MyDrive/valid_results.csv")

# # 2) Test 결과 저장
# test_preds_np, test_trues_np = test_preds.numpy(), test_targets.numpy()
# df_test = pd.DataFrame({
#     "D_lat_pred": test_preds_np[:, 0],
#     "D_lat_true": test_trues_np[:, 0],
#     "D_lon_pred": test_preds_np[:, 1],
#     "D_lon_true": test_trues_np[:, 1],
# })
# df_test.to_csv("/content/test_results.csv", index=False)
# df_test.to_csv("/content/drive/MyDrive/test_results.csv", index=False)
# print("✅ Saved: /content/test_results.csv & /content/drive/MyDrive/test_results.csv")


In [ ]:
# ================================
# 📊 Test Summary Statistics D_alt only)
# ================================
import numpy as np
import pandas as pd

# evaluate_on_test_set_direct 에서 나온 텐서(이미 CPU로 모임)
test_preds_np  = test_preds.numpy().reshape(-1)    # (N,1) 혹은 (N,) → (N,)
test_trues_np  = test_targets.numpy().reshape(-1)  # (N,1) 혹은 (N,) → (N,)

# 1) true/pred/error
df_test_stats = pd.DataFrame({
    "true_D_alt": test_trues_np,
    "pred_D_alt": test_preds_np,
})
df_test_stats["error_D_alt"] = df_test_stats["pred_D_alt"] - df_test_stats["true_D_alt"]
df_test_stats["squared_error"] = df_test_stats["error_D_alt"]**2
df_test_stats["abs_error"] = df_test_stats["error_D_alt"].abs()

print("\n==============================")
print("📋 D_alt Summary (true, pred, error)")
print(df_test_stats[["true_D_alt","pred_D_alt","error_D_alt"]].describe().round(6))

# 2) 단일 타깃이므로 전체 RMSE = sqrt(MSE)
overall_mse  = df_test_stats["squared_error"].mean()
overall_rmse = np.sqrt(overall_mse)
overall_mae  = df_test_stats["abs_error"].mean()

print(f"\n🎯 Overall MSE (from rows): {overall_mse:.6f}")
print(f"🎯 Overall RMSE: {overall_rmse:.6f}")
print(f"🎯 Overall MAE: {overall_mae:.6f}")
print(f"🎯 Overall MSE (from evaluate function): {test_metrics['MSE']:.6f}")

# (선택) 앞 10개 빠르게 확인
print("\n🔎 First 10 rows (true/pred/errors)")
print(df_test_stats[["true_D_alt","pred_D_alt","error_D_alt"]].head(10).round(6))



📋 D_alt Summary (true, pred, error)
        true_D_alt   pred_D_alt  error_D_alt
count  2100.000000  2100.000000  2100.000000
mean    -12.487619   -12.455767     0.031853
std      49.802773    44.250263    29.194197
min    -279.000000  -224.073166  -202.117523
25%     -20.000000   -13.372489   -16.075106
50%      -3.000000    -1.162371    -0.372397
75%      13.000000     7.799677    16.944136
max     160.000000   112.428391   162.919708

🎯 Overall MSE (from rows): 851.897522
🎯 Overall RMSE: 29.187284
🎯 Overall MAE: 21.397890
🎯 Overall MSE (from evaluate function): 851.897522

🔎 First 10 rows (true/pred/errors)
   true_D_alt  pred_D_alt  error_D_alt
0        -1.0   -3.495332    -2.495332
1        -6.0   -2.567490     3.432510
2         6.0  -26.483700   -32.483700
3        -9.0  -19.218481   -10.218483
4         6.0  -27.507858   -33.507858
5       -17.0  -15.260251     1.739749
6       -18.0  -33.516277   -15.516277
7        18.0  -18.608418   -36.608418
8        19.0   -7.294115   -2